In [1]:
import numpy as np
import random

import pickle as pkl
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from brainbox.behavior.training import compute_performance, plot_psychometric, plot_reaction_time
from brainwidemap import bwm_query, bwm_units, load_good_units, load_trials_and_mask
from tqdm import tqdm
from one.api import ONE
from brainbox.singlecell import bin_spikes2D
from iblatlas.regions import BrainRegions
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
from ibl_info.utils import check_config
from ibl_info.rsa_regression import (
    ideal_rsa_matrices,
    run_rsa_regression,
    simpler_rsa_matrices,
    run_rsa_regression_with_reward,
)
from sklearn.linear_model import LinearRegression
from scipy.spatial.distance import pdist, squareform
from scipy.stats import zscore
from ibl_info.manifold import (
    plot_pcas_separate_decomposition,
    plot_pcas_separate_decomposition_adapted,
)
import warnings

warnings.filterwarnings("ignore")

## Only correct trajectories

In [ ]:
with open("../data/generated/manifold/bwm_accumulated_data.pkl", "rb") as f:
    dx = pkl.load(f)

In [ ]:
# normalization = False
# model_vectors, model_names = ideal_rsa_matrices()

In [ ]:
plot_pcas_separate_decomposition_adapted(
    dx, "LGd", ["Left Con", "Left Incon", "Right Con", "Right Incon"]
)

In [ ]:
# now we fit kernels

In [ ]:
model_vectors, model_names, model_matrices = simpler_rsa_matrices()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, sharex=True, sharey=True)

for idx, (name, array) in enumerate(model_matrices.items()):
    cond_labels = ["Left_Con", "Left_Incon", "Right_Con", "Right_Incon"]
    if idx == 2:
        cbar = True
    else:
        cbar = False
    sns.heatmap(
        array,
        ax=ax[idx],
        linewidths=0.3,
        square=True,
        xticklabels=cond_labels,
        yticklabels=cond_labels,
        cbar=False,
    )
    ax[idx].set_title(f"{name}")

In [ ]:
results_correct_only = run_rsa_regression(
    dx, model_vectors=model_vectors, model_names=model_names, conditions=4, model_type="NNLS"
)

In [ ]:
from ibl_info.rsa_regression import plot_rsa_dynamics

In [ ]:
plot_rsa_dynamics(results_correct_only, "GRN", model_names)

In [ ]:
for k in results_correct_only.keys():
    plot_rsa_dynamics(results_correct_only, k, model_names)

In [ ]:
from ibl_info.rsa_regression import plot_rsa_summary_bars

In [ ]:
df = plot_rsa_summary_bars(results_correct_only, model_names)

## Manifold, all trials


In [ ]:
with open("../data/generated/manifold/bwm_manifold_8conditions.pkl", "rb") as f:
    dx_all = pkl.load(f)

In [ ]:
model_vectors, model_names, model_matrices = ideal_rsa_matrices()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=5, sharex=True, sharey=True)
cond_labels = [
    "L_Cong_Corr",
    "L_Cong_Err ",
    "L_Inc_Corr ",
    "L_Inc_Err  ",
    "R_Cong_Corr",
    "R_Cong_Err ",
    "R_Inc_Corr ",
    "R_Inc_Err  ",
]
for idx, (name, array) in enumerate(model_matrices.items()):

    if idx == 2:
        cbar = True
    else:
        cbar = False
    sns.heatmap(
        array,
        ax=ax[idx],
        linewidths=0.3,
        square=True,
        xticklabels=cond_labels,
        yticklabels=cond_labels,
        cbar=False,
    )
    ax[idx].set_title(f"{name}")

In [ ]:
results_all_without_outcome = run_rsa_regression(
    dx_all, model_vectors, ["Choice", "Prior", "Congruence", "Stimulus"], model_type="NNLS"
)

In [ ]:
results_all = run_rsa_regression(dx_all, model_vectors, model_names, model_type="NNLS")

In [ ]:
for k in results_correct_only.keys():
    plot_rsa_dynamics(results_all, k, model_names)

In [ ]:
plot_rsa_summary_bars(results_all, model_names)

In [ ]:
plot_rsa_summary_bars(results_all_without_outcome, ["Choice", "Prior", "Congruence", "Stimulus"])

## Manifold, all trials, with new regressors

In [6]:
with open("../data/generated/manifold/old_manifolds/bwm_manifold_8conditions.pkl", "rb") as f:
    dx_all = pkl.load(f)

In [7]:
model_vectors, model_names, model_matrices = ideal_rsa_matrices()